In [18]:
import torch
import torch.nn as nn
import math

In [19]:
class Embedder(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        
    def forward(self, x):
        return self.embed(x)

In [20]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype= torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)        
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

In [21]:
class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=5000):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model, max_len)

    def forward(self, x):
        x = self.embed(x)
        return self.pos(x)


In [22]:
vocab_size = 16000
d_model = 128

embedder = Embedder(vocab_size, d_model)

tokens = torch.tensor([[1, 42, 513]])
embeddings = embedder(tokens)

print(embeddings.shape)

torch.Size([1, 3, 128])


In [19]:
df.head()

,id,title,author,pub_date,genres,summary
0,620,Animal Farm,George Orwell,1945-08-17,"[""Roman à clef"", ""Satire"", ""Children's literat...","Old Major, the old boar on the Manor Farm, ca..."
1,843,A Clockwork Orange,Anthony Burgess,1962-01-01,"[""Science Fiction"", ""Novella"", ""Speculative fi...","Alex, a teenager living in near-future Englan..."
2,986,The Plague,Albert Camus,1947-01-01,"[""Existentialism"", ""Fiction"", ""Absurdist ficti...",The text of The Plague is divided into five p...
3,1756,An Enquiry Concerning Human Understanding,David Hume,NaN,[],The argument of the Enquiry proceeds by a ser...
4,2080,A Fire Upon the Deep,Vernor Vinge,NaN,"[""Hard science fiction"", ""Science Fiction"", ""S...",The novel posits that space around the Milky ...


In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses, InputExample
from sentence_transformers.util import cos_sim
import torch

# --- 1. Load and Prepare the Dataset ---
print("Step 1: Loading and Preparing the Dataset...")

df = pd.read_csv('../booksummaries/book_data_clean.csv')


# Make sure you have downloaded 'booksummaries.txt' and placed it in the same directory
# try:
#     df = pd.read_csv('booksummaries.txt', sep='\t', header=None, names=['wikipedia_id', 'freebase_id', 'title', 'author', 'publication_date', 'genres', 'summary'])
# except FileNotFoundError:
#     print("Error: 'booksummaries.txt' not found. Please download it from https://www.cs.cmu.edu/~dbamman/booksummaries.html")
#     exit()

# Clean up the data: drop rows with missing titles or summaries
df.dropna(subset=['title', 'summary'], inplace=True)
print(f"Dataset loaded with {len(df)} books after cleaning.")

# --- 2. Create Training Examples ---
print("\nStep 2: Creating Training Examples...")

# The model will learn to map the book title and its summary to a similar vector space.
# We will use MultipleNegativesRankingLoss, which requires positive pairs.
# A positive pair consists of two sentences that are semantically similar.
# Here, we use (title, summary) as a positive pair.
train_examples = []
for _, row in df.iterrows():
    train_examples.append(InputExample(texts=[row['title'], row['summary']]))

print(f"Created {len(train_examples)} training examples.")

# --- 3. Configure Model and Training ---
print("\nStep 3: Configuring Model and Training...")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device} 🚀")
if device == 'cpu':
    print("Warning: GPU not found. Training will be slow.")

# Load a pre-trained model. 'all-MiniLM-L6-v2' is a good starting point.
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name, device=device)

# Create a DataLoader to batch the training examples
# A batch size of 16 or 32 is a good starting point.
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Use MultipleNegativesRankingLoss, a great loss function for this task.
# It works by taking a batch of positive pairs and using the other items in the batch as negative examples.
train_loss = losses.MultipleNegativesRankingLoss(model)

# --- 4. Fine-Tune the Model ---
print("\nStep 4: Starting the Fine-Tuning Process...")

# Fine-tune the model. For a large dataset, this can take some time.
# We'll run it for one epoch in this example.
num_epochs = 1
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1) # 10% of train data for warm-up

model.fit(train_objectives=[(train_dataloader, train_loss)],
          epochs=num_epochs,
          warmup_steps=warmup_steps,
          output_path='./fine_tuned_book_embedder',
          show_progress_bar=True)

print("Model fine-tuning complete.")

# --- 5. Use the Fine-Tuned Model for Searching ---
print("\nStep 5: Using the Fine-Tuned Model for a Search Query...")

# Load your newly trained model
fine_tuned_model = SentenceTransformer('./fine_tuned_book_embedder')

# Embed all book summaries using the new model
# For efficiency, you should do this once and save the embeddings
book_summaries = df['summary'].tolist()
print("Encoding all book summaries with the new model... (This might take a while)")
book_embeddings = fine_tuned_model.encode(book_summaries, convert_to_tensor=True, show_progress_bar=True)

Step 1: Loading and Preparing the Dataset...
Dataset loaded with 16559 books after cleaning.

Step 2: Creating Training Examples...
Created 16559 training examples.

Step 3: Configuring Model and Training...

Step 4: Starting the Fine-Tuning Process...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,1.241600
1000,1.179300


Model fine-tuning complete.

Step 5: Using the Fine-Tuned Model for a Search Query...
Encoding all book summaries with the new model... (This might take a while)


Batches:   0%|          | 0/518 [00:00<?, ?it/s]


Top 5 results for the query: 'wizarding book with students'


TypeError: Cannot index by location index with a non-integer key

In [16]:
# --- Save embeddings (replace the old torch.save(book_embeddings, 'embedder.') cell) ---
import os, json
import torch
import numpy as np

os.makedirs('artifacts', exist_ok=True)

# Ensure tensor on CPU
emb_cpu = book_embeddings.detach().cpu()

meta = {
    "model_name": model_name if 'model_name' in globals() else "unknown",
    "num_items": int(emb_cpu.shape[0]),
    "dim": int(emb_cpu.shape[1]),
    "titles": df['title'].tolist(),
}

# Main packed file (tensor + meta)
torch.save({"embeddings": emb_cpu, "meta": meta}, "artifacts/book_embeddings.pt")
print("Saved artifacts/book_embeddings.pt")

# Tensor only (lightweight)
torch.save(emb_cpu, "artifacts/book_embeddings_tensor.pt")
print("Saved artifacts/book_embeddings_tensor.pt")

# Numpy format
np.save("artifacts/book_embeddings.npy", emb_cpu.numpy())
print("Saved artifacts/book_embeddings.npy")

# JSON metadata
with open("artifacts/book_embeddings_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Saved artifacts/book_embeddings_meta.json")

# Example load (later):
data = torch.load("artifacts/book_embeddings.pt", map_location="cpu")
emb = data["embeddings"]  # (N, D)
meta = data["meta"]
titles = meta["titles"]

Saved artifacts/book_embeddings.pt
Saved artifacts/book_embeddings_tensor.pt
Saved artifacts/book_embeddings.npy
Saved artifacts/book_embeddings_meta.json


In [19]:
user_query = "wizard"
query_embedding = fine_tuned_model.encode(user_query, convert_to_tensor=True).cpu()

# Find the 5 most similar books using cosine similarity
cosine_scores = cos_sim(query_embedding, emb)
top_results = torch.topk(cosine_scores[0], k=5)

print(f"\nTop 5 results for the query: '{user_query}'")
indices = top_results.indices.cpu().tolist()
scores = top_results.values.cpu().tolist()
for score, idx in zip(scores, indices):
    print(f"- {df['title'].iloc[idx]} (Score: {score:.4f})")


Top 5 results for the query: 'wizard'
- So You Want to Be a Wizard (Score: 0.5988)
- The Reign of the Brown Magician (Score: 0.5946)
- In the Empire of Shadow (Score: 0.5775)
- A Bad Spell in Yurt (Score: 0.5392)
- Equal Rites (Score: 0.5240)


In [14]:
import json, random, math, os, pandas as pd
from pathlib import Path
import sys

# --- Add tokenization utils (encode / decode) ---
sys.path.append(str(Path('..') / 'tokenization'))
from utils import encode, decode  # reuse existing tokenizer functions

# Device first so we can allocate directly there
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

# --- Load tokenizer assets ---
TOK_DIR = Path('..') / 'tokenization' / 'bpe_model'
with (TOK_DIR / 'merges.json').open('r', encoding='utf-8') as f:
    merges_data = json.load(f)
merges_list = merges_data.get('merges', [])  # (still unused here, encode() already encodes via replay in utils)
with (TOK_DIR / 'vocab.json').open('r', encoding='utf-8') as f:
    vocab_hex = json.load(f)
# id -> bytes
vocab_bytes = { int(k): bytes.fromhex(v) for k,v in vocab_hex.items() }

vocab_size_inferred = max(vocab_bytes.keys()) + 1
print('Vocab size inferred:', vocab_size_inferred)

# --- Load dataset summaries ---
DATA_CSV = Path('..') / 'booksummaries' / 'book_data_clean.csv'
df_embed = pd.read_csv(DATA_CSV)
summaries = df_embed['summary'].fillna('').astype(str).tolist()
print('Summaries:', len(summaries))

# --- Build training sequences (token id lists) using utils.encode ---
# Keep as Python lists for lightweight pair construction
encoded_sequences = [ encode(s) for s in summaries ]
if encoded_sequences:
    print('Example encoded length:', len(encoded_sequences[0]))
else:
    print('No sequences loaded; aborting.')

# --- Skip-gram with negative sampling (simple, GPU-leaning) ---
window = 2            # context window size on each side
neg_k = 5             # negatives per positive
embedding_dim = 128   # keep consistent with previous
max_pairs = 500000    # cap to avoid huge memory; adjust as needed

# Collect positive (center, context) pairs (CPU loop)
pairs = []
for seq in encoded_sequences:
    L = len(seq)
    for i, center in enumerate(seq):
        left = max(0, i - window)
        right = min(L, i + window + 1)
        for j in range(left, right):
            if j == i:
                continue
            pairs.append((center, seq[j]))
            if len(pairs) >= max_pairs:
                break
        if len(pairs) >= max_pairs:
            break
    if len(pairs) >= max_pairs:
        break
print('Positive pairs collected:', len(pairs))

if not pairs:
    raise ValueError('No training pairs collected; check data / parameters.')

# Convert pairs to GPU tensors once (centers, contexts) for faster batching
pairs_centers = torch.tensor([c for c,_ in pairs], device=device)
pairs_contexts = torch.tensor([ctx for _,ctx in pairs], device=device)
num_pairs = pairs_centers.shape[0]

# Frequency for negative sampling distribution (on device)
freq = torch.zeros(vocab_size_inferred, device=device)
for seq in encoded_sequences:  # still Python loop; could vectorize if needed
    for t in seq:
        if t < vocab_size_inferred:
            freq[t] += 1
prob = (freq ** 0.75)
prob = prob / prob.sum().clamp_min(1e-9)
# cumulative distribution (device)
cumprob = prob.cumsum(0)

# Model (two embeddings word2vec style) ---
embed_in = nn.Embedding(vocab_size_inferred, embedding_dim).to(device)
embed_ctx = nn.Embedding(vocab_size_inferred, embedding_dim).to(device)
optimizer = torch.optim.Adam(list(embed_in.parameters()) + list(embed_ctx.parameters()), lr=2e-3)
logsigmoid = nn.LogSigmoid()

batch_size = 1024
epochs = 1  # increase for better quality

# Vectorized negative sampler using cumulative distribution (expects cumprob on device)
def sample_neg(batch_sz: int):
    r = torch.rand(batch_sz, neg_k, device=device)
    return torch.searchsorted(cumprob, r)

for ep in range(epochs):
    # Shuffle order directly on GPU
    order = torch.randperm(num_pairs, device=device)
    total_loss = 0.0
    steps = 0
    for start in range(0, num_pairs, batch_size):
        batch_idx = order[start:start+batch_size]
        centers = pairs_centers[batch_idx]
        contexts = pairs_contexts[batch_idx]
        c_vec = embed_in(centers)
        ctx_vec = embed_ctx(contexts)
        pos_score = (c_vec * ctx_vec).sum(-1)
        neg_ids = sample_neg(centers.shape[0])          # (B, neg_k)
        neg_vec = embed_ctx(neg_ids)                    # (B, neg_k, D)
        neg_score = (c_vec.unsqueeze(1) * neg_vec).sum(-1)
        loss = - (logsigmoid(pos_score).mean() + logsigmoid(-neg_score).mean())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        steps += 1
        if steps % 100 == 0:
            print(f'epoch {ep+1} step {steps} loss {loss.item():.4f}')
    print(f'Epoch {ep+1} avg loss {total_loss/max(steps,1):.4f}')

# Final embedding matrix (average of input & context)
final_emb = (embed_in.weight.data + embed_ctx.weight.data) / 2

# --- Quick similarity helper ---
final_norm = final_emb / (final_emb.norm(dim=1, keepdim=True) + 1e-9)

def most_sim(tok_id, topk=5):
    v = final_norm[tok_id]
    sims = (final_norm @ v).detach().cpu()
    vals, idxs = sims.topk(topk+1)
    out = []
    for idx, val in zip(idxs.tolist(), vals.tolist()):
        if idx == tok_id:
            continue
        out.append((idx, round(val,4)))
        if len(out) >= topk:
            break
    return out

print('Nearest to token 1:', most_sim(1))

# Optionally save
os.makedirs('trained_embed', exist_ok=True)
torch.save({'weights': final_emb.cpu(), 'dim': embedding_dim}, 'trained_embed/embeddings.pt')
print('Saved to trained_embed/embeddings.pt')

NameError: name 'torch' is not defined